\*Please Run in colab

# Setup

### Environment Setup

In [1]:
!pip install instructor

In [2]:
!cd /content
!rm -rf macro_financial_forecasting

In [3]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1977, done.
remote: Counting objects: 100% (684/684), done.
remote: Compressing objects: 100% (240/240), done.
remote: Total 1977 (delta 523), reused 457 (delta 444), pack-reused 1293 (from 2)
Receiving objects: 100% (1977/1977), 39.64 MiB | 38.84 MiB/s, done.
Resolving deltas: 100% (1266/1266), done.


In [4]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Mount GDrive

In [5]:
# from google.colab import drive

# # This will prompt you to authorize Colab to access your Google Drive.
# drive.mount('/content/gdrive')
# GDRIVE_PATH = "/content/gdrive/MyDrive/macro_financial_forecasting_files/"

### Code Setup

In [6]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")
print(f"Config: {config}")

Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'https://feeds.bloomberg.com/real-e

# Agent

In [7]:
!pip install feedparser

In [8]:
!pip install langchain langchain-openai

In [8]:
# from typing import List, Dict, Any
# from datetime import datetime, timedelta, timezone
# import feedparser

# from langchain.tools import tool
# from langchain_openai import ChatOpenAI
# from langgraph.prebuilt import create_react_agent


In [ ]:
# @tool
# def get_bloomberg_rss_feeds(days: int = 1) -> List[Dict[str, str]]:
#     """Fetch Bloomberg RSS news items for the last N days."""

#     from config import Config
#     from data_model.bloomberg_news_entry import BloombergNewsEntry
#     config = Config()
#     feeds = config.rss_feeds[:2]

#     cutoff = datetime.now(timezone.utc) - timedelta(days=days)
#     news = []

#     for url in feeds:
#         feed = feedparser.parse(url)
#         for entry in feed.entries:
#             if hasattr(entry, "published_parsed"):
#                 published = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
#                 if published > cutoff:
#                     news.append({
#                         "Headline": entry.title,
#                         "Link": entry.link,
#                         "Article": entry.summary,
#                         "Date": published.isoformat(),
#                     })
#     return news #[BloombergNewsEntry.model_validate(record) for record in news]

In [ ]:
# from processor import NewsProcessor
# from pydantic import BaseModel

# _processor_instance = None

# def get_processor(config: Config) -> NewsProcessor:
#     global _processor_instance
#     if _processor_instance is None:
#         _processor_instance = NewsProcessor(config)
#     return _processor_instance

# class ProcessNewsInput(BaseModel):
#     data: List[BloombergNewsEntry]

# @tool(args_schema=ProcessNewsInput)
# def process_bloomberg_news(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
#     """
#     Run the full NewsProcessor pipeline on raw Bloomberg RSS feed entries.
#     Input: List of dicts with Headline, Link, Article, Date
#     Output: Processed dataframe converted to list[dict]
#     """
#     from config import Config
#     from processor import NewsProcessor

#     config = Config()
#     processor = get_processor(config)

#     # Pipeline
#     # data = processor.remove_redundant_info(data)
#     df = processor.enrich_news_entries_with_classifications(data)
#     df = processor.group_by_date_and_industry(df)
#     df = processor.filter_and_analyze_news(df)
#     df = processor.extract_impactful_news(df, top_n=3)
#     df = processor.get_consolidated_sentiment(df)
#     df = processor.get_explanation(df)

#     return df.to_dict(orient="records")

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
import json

from tools.data_feed import get_bloomberg_rss_feeds
from tools.data_processor import process_bloomberg_news
import nest_asyncio
nest_asyncio.apply()

async def run_agent():
  agent = create_react_agent(
      model="gpt-5-nano",
      tools=[get_bloomberg_rss_feeds, process_bloomberg_news]
  )
  instructions  = """Fetch Bloomberg RSS news from the past 1 day.
                    Then, process the news.
                    Return me json (list of dict)."""

  async for chunk in agent.astream({"messages": [{"role": "user", "content": instructions}]}, stream_mode="updates"):
    for step, data in chunk.items():
      print(f"step: {step}")
      print(f"content: {data['messages'][-1].content_blocks}")

  return data

data = await run_agent()

result = json.loads(data["messages"][-1].content_blocks[0]["text"])
print(f"Number of news entries: {len(result)}")


/tmp/ipython-input-320629513.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


step: agent
content: [{'type': 'tool_call', 'name': 'get_bloomberg_rss_feeds', 'args': {'days': 1}, 'id': 'call_I5A97TzKu1Dcj4fl41ueaPfq'}]
step: tools
content: [{'type': 'text', 'text': '[{"Headline": "Asian Stocks to Ebb as Global Equity Rally Stalls: Markets Wrap", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/asian-stocks-to-ebb-as-global-equity-rally-stalls-markets-wrap", "Article": "Asian stocks were set for a muted open Friday as a sharp rebound in global equities over the past week began to stall.", "Date": "2025-11-27T22:29:53+00:00"}, {"Headline": "Emerging Assets Halt Rally in Quiet Session Amid US Holiday", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/china-sets-tone-as-rebound-in-emerging-market-assets-pauses", "Article": "Emerging-market assets halted their recent advance on Thursday, with a gauge of currencies ending the day little changed and stocks falling slightly amid thin liquidity due to the Thanksgiving holiday.", "Date": "2025-11-27T1

In [ ]:
result[10]

{'Industry': 'Energy',
 'Date': '2025-11-27T12:39:44+00:00',
 'News': [{'Headline': 'Nigeria’s NNPC, Heirs Restore Gas Facility, Doubling Output',
   'Date': '2025-11-27T12:39:44+00:00',
   'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/nigeria-s-nnpc-heirs-restore-gas-facility-doubling-output',
   'Article': 'Nigerian National Petroleum Co. and Heirs Energies have restored production at a gas facility that had been offline for more than a year, enabling the joint venture to double output and increase supplies to power producers struggling with fuel shortages.',
   'SentimentScore': -0.9157842397689819,
   'Industry': 'Energy'}],
 'ArticleCount': 1,
 'ImpactfulNews': [{'Headline': 'Nigeria’s NNPC, Heirs Restore Gas Facility, Doubling Output',
   'Article': 'Nigerian National Petroleum Co. and Heirs Energies have restored production at a gas facility that had been offline for more than a year, enabling the joint venture to double output and increase supplies to power produc